# Archive

In [ ]:
file_path = "test"

import json
import os


def streak_data():
    base_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), "../../"))
    json_file_path = os.path.join(base_dir, file_path)


streak_data = streak_data()

if streak_data:
    print(json.dumps(streak_data, indent=4))

## Free Non API Text to Audio

In [ ]:
from gtts import gTTS
from gtts.tts import gTTSError


def text_to_speech_gtts_mp3(text, language="en", mp3_filename="output.mp3"):
    try:
        tts = gTTS(text=text, lang=language, slow=False)
        tts.save(mp3_filename)
        return mp3_filename
    except gTTSError as e:
        print(f"Error during text-to-speech conversion: {e}")
        return None


text_to_convert = (
    "HELLO THERE! Welcome Back! ... In which skill do you want to level up now?"
)
output_file_name = "my_audio.mp3"
mp3_file = text_to_speech_gtts_mp3(text_to_convert, mp3_filename=output_file_name)

if mp3_file:
    print(f"MP3 file saved successfully as: {mp3_file}")
else:
    print("Failed to create MP3 file.")

## Gemini One Time Message Conversation

In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("Google_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash-lite-preview-02-05")
response = model.generate_content("Tell me about gojo satoru?")
print(response.text)

## Gemini Normal Conversation with Previous History

In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("Google_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash-lite-preview-02-05")

conversation_history = ""


def get_response(prompt):
    global conversation_history

    conversation_history += f"User: {prompt}\n"

    full_prompt = conversation_history + f"Bot: "
    response = model.generate_content(full_prompt)

    bot_response = response.text
    if "Bot:" in bot_response:
        bot_response = bot_response.split("Bot:")[1].strip()

    conversation_history += f"Bot: {bot_response}\n"

    return bot_response


while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break

    bot_response = get_response(user_input)
    print("user:", user_input)
    print("Bot:", bot_response)

## Gemini Normal Conversation with Previous History and special data piece

In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("Google_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash-lite-preview-02-05")

context = """
* First Name: Katoro
* Last Name: Kamado
* Age: 8
* Nationality: Japanese
"""

conversation_history = context + "\n"


def get_response(prompt):
    global conversation_history

    conversation_history += f"User: {prompt}\n"

    full_prompt = conversation_history + f"Bot: "
    response = model.generate_content(full_prompt)

    bot_response = response.text
    if "Bot:" in bot_response:
        bot_response = bot_response.split("Bot:")[1].strip()

    conversation_history += f"Bot: {bot_response}\n"

    return bot_response


while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break

    bot_response = get_response(user_input)
    print("user:", user_input)
    print("Bot:", bot_response)

## Gemini Normal Conversation with Previous History and RAG

### Creates Vector DB

In [ ]:
import os
import shutil
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb


def delete_files_and_subfolders(folder_path):
    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' does not exist.")
        return

    try:
        shutil.rmtree(folder_path)
        print(f"Deleted: {folder_path} and all its contents.")
    except Exception as e:
        print(f"Error deleting {folder_path}: {e}")


def load_and_store_pdfs_in_chroma(
    data_path, chroma_path, collection_name="psychologist"
):
    delete_files_and_subfolders(chroma_path)

    if not os.path.exists(chroma_path):
        os.makedirs(chroma_path)

    os.chmod(chroma_path, 0o777)

    chroma_client = chromadb.PersistentClient(path=chroma_path)

    try:
        chroma_client.delete_collection(name=collection_name)
    except Exception:
        print("No existing collection to delete.")
    collection = chroma_client.get_or_create_collection(name=collection_name)

    loader = PyPDFDirectoryLoader(data_path)
    raw_documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=100,
        length_function=len,
        is_separator_regex=False,
    )

    chunks = text_splitter.split_documents(raw_documents)

    documents = [chunk.page_content for chunk in chunks]
    ids = [f"ID{i}" for i, _ in enumerate(chunks)]
    metadata = [chunk.metadata for chunk in chunks]

    collection.upsert(
        documents=documents,
        metadatas=metadata,
        ids=ids,
    )

    print("Data successfully added to ChromaDB.")


DATA_PATH = "Database\\AI_Database\\RAG_Database"
CHROMA_PATH = "Database\\AI_Database\\Vector_Database"

load_and_store_pdfs_in_chroma(DATA_PATH, CHROMA_PATH)

### Conversation Single Time

In [ ]:
import chromadb
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()

CHROMA_PATH = "Database\\AI_Database\\Vector_Database"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name="psychologist")

user_query = input("How are you feeling today? I'm here to listen.\n\n")

results = collection.query(query_texts=[user_query], n_results=1)

genai.configure(api_key=os.getenv("Google_API_KEY"))

model = genai.GenerativeModel(
    "gemini-2.0-flash-lite-preview-02-05",
    system_instruction=f"""
    You are a compassionate and deeply empathetic psychologist. Your goal is to provide a safe space where the user feels heard, understood, and supported. 
    Respond in a warm and non-judgmental manner, offering emotional validation and gentle guidance.
    
    Always prioritize active listening, use open-ended questions to encourage reflection, and offer comfort when needed.
    If relevant, provide simple mindfulness or grounding techniques to help the user feel more at ease.
    
    Strictly answer ONLY based on the provided context.
    Context: {results['documents']}
    """,
)

response = model.generate_content(user_query)

print("\n\n---------------------\n\n")
print(response.text)

### Conversation with pervious History

In [ ]:
import chromadb
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()

CHROMA_PATH = "Database/AI_Database/Vector_Database"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name="psychologist")

genai.configure(api_key=os.getenv("Google_API_KEY"))

model = genai.GenerativeModel("gemini-2.0-flash-lite-preview-02-05")

conversation_history = ""


def get_response(user_query):
    global conversation_history

    results = collection.query(query_texts=[user_query], n_results=1)
    context = results["documents"][0] if results["documents"] else ""

    conversation_history += f"User: {user_query}\n"

    full_prompt = (
        f"You are a compassionate and deeply empathetic psychologist. "
        f"Your goal is to provide a safe space where the user feels heard, understood, and supported. "
        f"Respond in a warm and non-judgmental manner, offering emotional validation and gentle guidance.\n\n"
        f"Context from previous conversations: {context}\n\n"
        f"{conversation_history}Bot: "
    )

    response = model.generate_content(full_prompt)
    bot_response = response.text.strip()

    conversation_history += f"Bot: {bot_response}\n"

    return bot_response


while True:
    user_input = input("How are you feeling today? I'm here to listen.\n\n")
    if user_input.lower() in ["exit", "quit"]:
        break

    bot_response = get_response(user_input)
    print("\n\n---------------------\n\n")
    print(bot_response)

## Speech to Text

In [ ]:
import speech_recognition as sr


def convert_wav_to_text(wav_file_path):
    recognizer = sr.Recognizer()

    with sr.AudioFile(wav_file_path) as source:
        audio_data = recognizer.record(source)

    try:
        text = recognizer.recognize_google(audio_data)
        return text
    except sr.UnknownValueError:
        return "Speech Recognition could not understand the audio."
    except sr.RequestError as e:
        return f"Could not request results from Speech Recognition service; {e}"


if __name__ == "__main__":
    wav_file_path = "Recording.wav"
    result = convert_wav_to_text(wav_file_path)
    print("Converted text:")
    print(result)

## .mp3 to .wav

In [ ]:
from pydub import AudioSegment
import subprocess


def check_ffmpeg():
    try:
        subprocess.check_output(["ffmpeg", "-version"])
    except FileNotFoundError:
        print("ffmpeg is not installed or not in your system's PATH.")
        return False
    return True


def mp3_to_wav(mp3_file, wav_file):
    if check_ffmpeg():
        sound = AudioSegment.from_mp3(mp3_file)
        sound.export(wav_file, format="wav")
    else:
        print("Cannot convert MP3 to WAV. ffmpeg is required.")


mp3_path = "Test_Recording.mp3"
wav_path = "Test_output.wav"

mp3_to_wav(mp3_path, wav_path)

## Increasing Playback Speed

In [ ]:
from pydub import AudioSegment
from pydub.effects import speedup


def increase_playback_speed_no_pitch(input_file, output_file, speed_factor):
    sound = AudioSegment.from_mp3(input_file)
    new_sound = speedup(sound, playback_speed=speed_factor)
    new_sound.export(output_file, format="mp3")


input_file = "Test_Recording.mp3"
output_file = "Test_output_speed_up.mp3"
speed_factor = 1.3

increase_playback_speed_no_pitch(input_file, output_file, speed_factor)